# Module 10 — Frozen-Equilibrium Shift Diagnostics

Diagnostic test of the hypothesis that the 2025 drawdown was driven by stale pair equilibria rather than broad market beta. The module keeps each pair's original alpha, beta and mu frozen and asks whether the realized residual spread developed a persistent rolling-mean displacement from that formation equilibrium.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from src.backtest import run_walk_forward_backtest, compute_log_spread

pd.set_option('display.max_columns', 120)
OUT = Path('data/processed/equilibrium_shift')
OUT.mkdir(parents=True, exist_ok=True)

## 1. Reproduce original static OOS strategy

In [ ]:
train_prices = pd.read_parquet('data/processed/train_prices.parquet')
test_prices = pd.read_parquet('data/processed/test_prices.parquet')
eligible_pairs = pd.read_parquet('data/processed/eligible_pairs.parquet')
cointegrated_pairs = pd.read_parquet('data/processed/cointegrated_pairs.parquet')
rf_data = pd.read_parquet('data/processed/risk_free_rates.parquet')
risk_free_rates = rf_data.iloc[:,0] if isinstance(rf_data,pd.DataFrame) else pd.Series(rf_data)
risk_free_rates.index = pd.to_datetime(risk_free_rates.index)
risk_free_rates = risk_free_rates.sort_index().astype(float)

results = run_walk_forward_backtest(
    train_prices=train_prices, test_prices=test_prices, eligible_pairs=eligible_pairs,
    cointegrated_pairs=cointegrated_pairs, risk_free_rates=risk_free_rates,
    initial_capital=100000.0, entry_z=1.5, target_probability=0.70,
    memory_window=60, max_horizon_days=126, n_paths=5000, ewma_lambda=0.94, seed=42)
trades = results['trades'].copy()
equity = results['equity_curve'].copy()
eq = equity['equity'].astype(float)
dd = eq/eq.cummax()-1
trough = dd.idxmin(); peak = eq.loc[:trough].idxmax()
print('Peak:', peak, 'Trough:', trough, 'Max DD:', float(dd.loc[trough]))

## 2. Identify pairs most exposed to the maximum-drawdown interval

In [ ]:
trades['entry_date']=pd.to_datetime(trades['entry_date'])
trades['exit_date']=pd.to_datetime(trades['exit_date'])
overlap = trades[(trades['entry_date']<=trough) & (trades['exit_date']>=peak)].copy()
pair_dd = (overlap.groupby('pair').agg(
    n_overlap_trades=('pair','size'), total_realized_pnl=('pnl','sum'),
    mean_trade_return=('trade_return','mean'), win_rate=('pnl',lambda x:(x>0).mean()))
    .sort_values('total_realized_pnl').reset_index())
display(pair_dd.head(15))
pair_dd.to_parquet(OUT/'drawdown_pair_contributions.parquet',index=False)
focus_pairs = pair_dd.head(10)['pair'].tolist()
print('Focus pairs:', focus_pairs)

## 3. Frozen spread versus rolling realized equilibrium

In [ ]:
if 'pair' not in cointegrated_pairs.columns:
    cointegrated_pairs = cointegrated_pairs.copy()
    cointegrated_pairs['pair']=cointegrated_pairs['dependent'].astype(str)+'-'+cointegrated_pairs['independent'].astype(str)
if 'pair' not in eligible_pairs.columns:
    eligible_pairs = eligible_pairs.copy()
    eligible_pairs['pair']=eligible_pairs['dependent'].astype(str)+'-'+eligible_pairs['independent'].astype(str)
frozen = eligible_pairs.merge(cointegrated_pairs[['pair','alpha','beta']],on='pair',how='left',validate='one_to_one')
full_prices = pd.concat([train_prices,test_prices]).sort_index()
ROLL = 63
rows=[]; series_store={}
for pair in focus_pairs:
    r=frozen.loc[frozen['pair']==pair].iloc[0]
    s=compute_log_spread(full_prices,r['dependent'],r['independent'],r['alpha'],r['beta']).loc[test_prices.index]
    mu=float(r['mu']); var=float(r['variance'])
    rm=s.rolling(ROLL,min_periods=ROLL).mean()
    disp=rm-mu
    zmean=disp/np.sqrt(var)
    dd_slice=disp.loc[peak:trough].dropna()
    pre_slice=disp.loc[:peak].tail(126).dropna()
    post_slice=disp.loc[trough:].dropna()
    rows.append({
        'pair':pair,'frozen_mu':mu,
        'pre_peak_mean_displacement':pre_slice.mean(),
        'drawdown_mean_displacement':dd_slice.mean(),
        'drawdown_mean_abs_displacement':dd_slice.abs().mean(),
        'drawdown_mean_abs_z_of_rolling_mean':zmean.loc[peak:trough].abs().mean(),
        'post_trough_mean_displacement':post_slice.mean(),
        'share_drawdown_days_same_sign':max((dd_slice>0).mean(),(dd_slice<0).mean()) if len(dd_slice) else np.nan
    })
    series_store[pair]=(s,rm,mu,zmean)
shift_metrics=pd.DataFrame(rows).sort_values('drawdown_mean_abs_z_of_rolling_mean',ascending=False)
display(shift_metrics)
shift_metrics.to_parquet(OUT/'equilibrium_shift_metrics.parquet',index=False)

## 4. Visual inspection of the strongest displacement pairs

In [ ]:
plot_pairs=shift_metrics.head(6)['pair'].tolist()
fig,axes=plt.subplots(len(plot_pairs),1,figsize=(13,3*len(plot_pairs)),sharex=True)
if len(plot_pairs)==1: axes=[axes]
for ax,pair in zip(axes,plot_pairs):
    s,rm,mu,zmean=series_store[pair]
    s.plot(ax=ax,label='Frozen residual spread',alpha=.55)
    rm.plot(ax=ax,label=f'{ROLL}d rolling mean',linewidth=2)
    ax.axhline(mu,linestyle='--',label='Frozen mu')
    ax.axvspan(peak,trough,alpha=.10)
    ax.set_title(pair)
    ax.legend(loc='best')
plt.tight_layout(); plt.show()

## 5. Residual stationarity around the drawdown

In [ ]:
adf_rows=[]
for pair in focus_pairs:
    s,_,_,_=series_store[pair]
    for label,end in [('pre_peak',peak),('trough',trough)]:
        sample=s.loc[:end].tail(126).dropna()
        if len(sample)>=50:
            stat,p,*_=adfuller(sample,autolag='AIC')
            adf_rows.append({'pair':pair,'window':label,'end_date':end,'n_obs':len(sample),'adf':stat,'pvalue':p})
adf_table=pd.DataFrame(adf_rows)
display(adf_table.pivot(index='pair',columns='window',values='pvalue'))
adf_table.to_parquet(OUT/'local_adf_diagnostics.parquet',index=False)

## Interpretation

The diagnostic supports an equilibrium-shift mechanism when losing drawdown pairs show a rolling residual mean that remains persistently displaced from the frozen formation mu, especially when the displacement is large relative to the formation stationary standard deviation and local ADF evidence weakens. This does not by itself prove a structural break; it demonstrates that the static strategy may have been treating a moving relationship as a temporary deviation from a fixed equilibrium.